# Named Entity Recognition

This notebook covers Named Entity Recognition (NER), a fundamental task in natural language processing that involves identifying and classifying named entities in text into predefined categories such as person names, organisations, locations, dates, monetary values, and more.

NER is essential for many real-world applications including information extraction, question answering, and document summarisation.

In [ ]:
# Import required libraries
import spacy

# Load English model
nlp = spacy.load('en_core_web_sm')

print("spaCy model loaded successfully.")
print(f"Model: en_core_web_sm")
print(f"Default pipeline components: {nlp.pipe_names}")

## 1. Understanding Named Entity Recognition

NER involves identifying spans of text that refer to specific entities and classifying them into categories. The most common entity types are:

- **PERSON**: People, including fictional
- **ORG**: Companies, agencies, institutions
- **GPE**: Countries, cities, states
- **DATE**: Dates, durations
- **MONEY**: Monetary values
- **PERCENT**: Percentages
- **FAC**: Buildings, airports, etc.
- **PRODUCT**: Objects, vehicles, foods
- **EVENT**: Named hurricanes, battles, wars
- **WORK_OF_ART**: Titles of books, songs, etc.

Modern NER systems use sequence labelling models, often based on transformers.

In [ ]:
# Sample text for NER demonstration
text = """
Apple Inc. is planning to open a new store in London on December 15, 2024.
The company, led by CEO Tim Cook, will invest $50 million in the new location.
This is part of Apple's expansion strategy in Europe, following successful launches 
in Paris and Berlin earlier this year.
"""

# Process the text
doc = nlp(text)

print("=== Sample Text ===")
print(text)

In [ ]:
# Display named entities
print("=== Named Entities Found ===\n")

print(f"{'Entity':<20} {'Label':<15} {'Start':<8} {'End':<8}")
print("-" * 55)

for ent in doc.ents:
    print(f"{ent.text:<20} {ent.label_:<15} {ent.start_char:<8} {ent.end_char:<8}")

In [ ]:
# Visualise entities in context
print("=== Entities with Context ===\n")

for ent in doc.ents:
    # Get surrounding context
    start = max(0, ent.start_char - 20)
    end = min(len(text), ent.end_char + 20)
    context = text[start:end].replace('\n', ' ')
    
    print(f"Entity: '{ent.text}'")
    print(f"Type: {ent.label_}")
    print(f"Context: ...{context}...")
    print()

## 2. Using spaCy for NER

spaCy provides a powerful and efficient NER system. Let's explore its capabilities in more detail.

In [ ]:
# More complex example
text2 = """
On January 15, 2024, Microsoft announced a partnership with OpenAI to develop 
advanced AI solutions. The deal worth $10 billion will be invested over the 
next five years. Satya Nadella, CEO of Microsoft, stated that this collaboration 
will revolutionise the technology industry. The new AI models will be available 
through Azure cloud services starting from Q3 2024.
"""

doc2 = nlp(text2)

print("=== Second Example ===")
print(text2)
print("\n" + "="*50)

# Group entities by type
entity_types = {}
for ent in doc2.ents:
    if ent.label_ not in entity_types:
        entity_types[ent.label_] = []
    entity_types[ent.label_].append(ent.text)

print("\n=== Entities by Type ===")
for label, entities in entity_types.items():
    print(f"\n{label}:")
    for e in entities:
        print(f"  - {e}")

In [ ]:
# Access token-level annotations
print("=== Token-Level Analysis ===\n")

print(f"{'Token':<15} {'POS':<10} {'Tag':<10} {'Entity':<15} {'Label'}")
print("-" * 70)

for token in doc2:
    ent = token.ent_type_ if token.ent_type_ else ""
    ent_iob = token.ent_iob_ if token.ent_iob_ != 'O' else ""
    label = f"{ent_iob}-{ent}" if ent else ""
    print(f"{token.text:<15} {token.pos_:<10} {token.tag_:<10} {ent:<15} {label}")

## 3. Custom NER with spaCy

spaCy allows training custom NER models for domain-specific entities.

In [ ]:
# Create training data for custom NER
# In practice, you'd have a much larger dataset

TRAIN_DATA = [
    ("Google was founded by Larry Page and Sergey Brin.", {
        "entities": [(0, 6, "ORG"), (25, 35, "PERSON"), (40, 52, "PERSON")]
    }),
    ("Apple's headquarters is in Cupertino.", {
        "entities": [(0, 5, "ORG"), (26, 35, "GPE")]
    }),
    ("Microsoft CEO Satya Nadella spoke today.", {
        "entities": [(0, 10, "ORG"), (14, 26, "PERSON")]
    }),
    ("Amazon offers cloud services via AWS.", {
        "entities": [(0, 7, "ORG"), (28, 31, "PRODUCT")]
    }),
]

print("=== Custom NER Training Data ===")
print("Format: (text, {'entities': [(start, end, label), ...]})")
print("\nExample:")
for text, annotations in TRAIN_DATA[:2]:
    print(f"\nText: {text}")
    print(f"Entities: {annotations['entities']}")

In [ ]:
# Note: Training a custom NER model requires significant data and time
# Here's how you would configure the training process

print("=== Custom NER Training Configuration ===")
print("""
To train a custom NER model in spaCy:

```python
import spacy
from spacy.training import Example

# Create a blank model
nlp = spacy.blank('en')

# Add the NER component
ner = nlp.add_pipe('ner')

# Add labels
for label in ['ORG', 'PERSON', 'PRODUCT']:
    ner.add_label(label)

# Convert training data
train_examples = []
for text, annotations in TRAIN_DATA:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    train_examples.append(example)

# Train the model
optimizer = nlp.begin_training()
for i in range(20):
    losses = {}
    for batch in spacy.util.minibatch(train_examples, size=2):
        nlp.update(batch, sgd=optimizer, losses=losses)
    print(f"Iteration {i}, Losses: {losses}")
```

Note: This requires a larger training dataset for good performance.
""")

## 4. Using Transformers for NER

For better accuracy, especially on complex text, we can use transformer-based models like BERT.

In [ ]:
from transformers import pipeline

# Load a pre-trained NER model from HuggingFace
ner_transformer = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("=== Transformer-based NER ===")
print("Model: dslim/bert-base-NER")

In [ ]:
# Test with the same text
results = ner_transformer(text2)

print("=== NER Results ===\n")
print(f"{'Entity':<20} {'Type':<15} {'Score'}")
print("-" * 50)
for result in results:
    print(f"{result['word']:<20} {result['entity_group']:<15} {result['score']:.4f}")

## 5. Comparing spaCy and Transformers

Let's compare the results from both approaches.

In [ ]:
# Compare results
print("=== Comparison: spaCy vs Transformers ===\n")

print("spaCy Results:")
for ent in doc2.ents:
    print(f"  {ent.text} -> {ent.label_}")

print("\nTransformer Results:")
for r in results:
    print(f"  {r['word']} -> {r['entity_group']}")

print("\n=== Key Differences ===")
print("""
spaCy (en_core_web_sm):
  - Faster inference
  - Good for general NER
  - Smaller model size
  - May miss some entity types

Transformers (bert-base-NER):
  - Higher accuracy
  - Better at fine-grained entities
  - Slower inference
  - Requires more resources
""")

## 6. Practical Applications

NER has many practical applications in real-world scenarios.

In [ ]:
# Example: Extracting information from news articles
news_article = """
Tech giant Google announced on March 15, 2024 that it will acquire 
startup company DeepMind for $500 million. The acquisition is expected 
to close by Q4 2024. Google CEO Sundar Pichai commented on the deal.
"""

doc_news = nlp(news_article)

print("=== Practical Application: News Extraction ===\n")
print(f"Article: {news_article.strip()}\n")

# Extract entities
entities = {}
for ent in doc_news.ents:
    if ent.label_ not in entities:
        entities[ent.label_] = []
    entities[ent.label_].append(ent.text)

print("Extracted Information:")
for label, values in entities.items():
    print(f"  {label}: {', '.join(values)}")

## Summary

In this notebook, we covered Named Entity Recognition:

1. **Understanding**: Learned about NER concepts and common entity types
2. **spaCy NER**: Used spaCy for efficient named entity recognition
3. **Token Analysis**: Explored token-level entity annotations
4. **Custom NER**: Learned how to train custom NER models
5. **Transformers**: Used BERT-based NER for higher accuracy
6. **Comparison**: Compared spaCy and transformer-based approaches
7. **Applications**: Explored practical NER applications

NER is a crucial component of many NLP pipelines. The choice between spaCy and transformers depends on your specific requirements for speed, accuracy, and resource availability.